In [46]:
import tiktoken
import torch
from torch.utils.data import Dataset, DataLoader
import pandas as pd
from torchvision.transforms import ToTensor
import numpy as np
from sklearn.model_selection import train_test_split



In [47]:
gpt2 = tiktoken.get_encoding("gpt2")
dataset_json = "dataset.json" #Ruta al json con el dataset

encoder = tiktoken.Encoding(
    name = "encoder",
    pat_str = gpt2._pat_str,
    mergeable_ranks = gpt2._mergeable_ranks,
    special_tokens = {
        **gpt2._special_tokens,
        "<START>": len(gpt2._mergeable_ranks) +1,
        "<END>": len(gpt2._mergeable_ranks) +2,
        "<PAD>": len(gpt2._mergeable_ranks) +3
    }
)

In [ ]:
CONTEXT_LENGTH = 300

dataset = pd.read_json(dataset_json)

df_train,df_val = train_test_split(dataset,test_size=0.2,random_state=42,shuffle=True)

# Dataset
class Dataset(Dataset):
    def __init__(self, data):
        self.data = data
        self.X = self.data['en']
        self.y = self.data['es']
        

    def __len__(self):
        return len(self.data)
    
    def adjust_length(self,text):
        pad = [encoder._special_tokens["<PAD>"] for _ in range(CONTEXT_LENGTH-len(text))]
        return np.concatenate((text,pad))
    

    def __getitem__(self, idx):

        en_text = encoder.encode(self.X.iloc[idx])
        es_text = encoder.encode(self.y.iloc[idx])


        #Truncar entrada en caso de que se supere dimension entrada x_n
        if len(en_text) > CONTEXT_LENGTH:
            en_text =  en_text[:300]
        
        if len(es_text) > CONTEXT_LENGTH:
            es_text = es_text[:(CONTEXT_LENGTH-2)]


        en_text = self.adjust_length(en_text)
        es_text = self.adjust_length(es_text)

       
        #Añadimos tokens especiales al targe
        es_text = np.concatenate(([encoder._special_tokens["<START>"]],
                                  es_text[:(CONTEXT_LENGTH-2)],
                                  [encoder._special_tokens["<END>"]]),
                                  axis=0)


        #COnvertimos a tensores
        es_text =  torch.tensor(es_text,dtype=torch.long)
        en_text = torch.tensor(en_text,dtype= torch.long)

        return en_text, es_text

# Crear dataset y dataloader
train_dataset = Dataset(df_train)
train_dataloader = DataLoader(train_dataset, batch_size=32,shuffle=True)

for X_batch, y_batch in train_dataloader:
    print(X_batch)
    print(y_batch)
    break

800000
tensor([[10995,    11,  8169,  ..., 50259, 50259, 50259],
        [ 6187, 10461,    11,  ..., 50259, 50259, 50259],
        [ 3856,   616,  8319,  ..., 50259, 50259, 50259],
        ...,
        [   40,  2626,   790,  ..., 50259, 50259, 50259],
        [   47, 40225,   307,  ..., 50259, 50259, 50259],
        [ 1212,   318,  1583,  ..., 50259, 50259, 50259]])
tensor([[50257,    50,  8836,  ..., 50259, 50259, 50258],
        [50257,  2782,    72,  ..., 50259, 50259, 50258],
        [50257,    47, 10205,  ..., 50259, 50259, 50258],
        ...,
        [50257,  5990,    67,  ..., 50259, 50259, 50258],
        [50257,  2348,   397,  ..., 50259, 50259, 50258],
        [50257,  9527,  1583,  ..., 50259, 50259, 50258]])
